In [1]:
import polars as pl

In [2]:
inicial = (
    pl.scan_parquet(r"C:\Users\Usuario\Downloads\covid_rj.parquet")

    .select([
        "paciente_idade",
        "paciente_enumSexoBiologico",
        "paciente_endereco_nmMunicipio",
        "paciente_endereco_uf",
        "vacina_dataAplicacao",
        "vacina_fabricante_nome",
        "vacina_categoria_nome"
    ])

    .with_columns([
        pl.col("paciente_idade").cast(pl.Int64, strict=False),

        pl.col("vacina_dataAplicacao")
        .str.to_date("%Y-%m-%d", strict=False),

        pl.lit("RJ").alias("estado"),

        pl.lit("Sudeste").alias("regiao")
    ])

    .filter(
        pl.col("paciente_idade").is_not_null() &
        (pl.col("paciente_idade") >= 0) &
        (pl.col("paciente_idade") <= 120) &
        pl.col("paciente_endereco_nmMunicipio").is_not_null() &
        pl.col("paciente_endereco_uf").is_not_null()
    )
)

display(inicial.head(5).collect())

paciente_idade,paciente_enumSexoBiologico,paciente_endereco_nmMunicipio,paciente_endereco_uf,vacina_dataAplicacao,vacina_fabricante_nome,vacina_categoria_nome,estado,regiao
i64,str,str,str,date,str,str,str,str
1,"""F""","""SAO JOSE DO VALE DO RIO PRETO""","""RJ""",2024-05-15,"""BUTANTAN""","""Faixa Etária""","""RJ""","""Sudeste"""
56,"""F""","""MAGE""","""RJ""",2022-02-16,"""JANSSEN""","""Faixa Etária""","""RJ""","""Sudeste"""
34,"""F""","""RIO DE JANEIRO""","""RJ""",2022-04-04,"""JANSSEN""","""Faixa Etária""","""RJ""","""Sudeste"""
54,"""M""","""SAO GONCALO""","""RJ""",2021-07-27,"""ASTRAZENECA/FIOCRUZ""","""Comorbidades""","""RJ""","""Sudeste"""
30,"""F""","""PETROPOLIS""","""RJ""",2021-08-13,"""PFIZER""","""Faixa Etária""","""RJ""","""Sudeste"""


In [3]:
df_municipios = (
    inicial
    .group_by(
        "paciente_endereco_nmMunicipio",
        "paciente_endereco_uf",
        "estado",
        "regiao"
    )

    .agg([
        pl.len().alias("total_vacinacoes"),

        pl.col("paciente_idade")
        .mean()
        .round(2)
        .alias("media_idade")
    ])

    .rename({
        "paciente_endereco_nmMunicipio": "municipio",
        "paciente_endereco_uf": "uf"
    })
)

display(df_municipios.head(10).collect())

municipio,uf,estado,regiao,total_vacinacoes,media_idade
str,str,str,str,u32,f64
"""NOVA SERRANA""","""MG""","""RJ""","""Sudeste""",88,34.99
"""HOLAMBRA""","""SP""","""RJ""","""Sudeste""",20,44.95
"""PIRES FERREIRA""","""CE""","""RJ""","""Sudeste""",378,34.13
"""OEIRAS""","""PI""","""RJ""","""Sudeste""",21,34.62
"""MUCUM""","""RS""","""RJ""","""Sudeste""",7,40.71
"""SENADOR ALEXANDRE COSTA""","""MA""","""RJ""","""Sudeste""",15,34.27
"""BOCAINA""","""PI""","""RJ""","""Sudeste""",1,18.0
"""ANTA GORDA""","""RS""","""RJ""","""Sudeste""",2,19.0
"""SANTANA DO GARAMBEU""","""MG""","""RJ""","""Sudeste""",15,29.8


In [4]:
df_agregado = df_municipios.collect()

In [5]:
q1 = (
    df_agregado
    .select(
        pl.col("total_vacinacoes")
        .quantile(0.25)
    )
    .item()
)

q3 = (
    df_agregado
    .select(
        pl.col("total_vacinacoes")
        .quantile(0.75)
    )
    .item()
)

print(f"Municípios abaixo de {q1:.0f} vacinações são 'Baixa Vacinação'")
print(f"Municípios acima de {q3:.0f} vacinações são 'Alta Vacinação'")

Municípios abaixo de 4 vacinações são 'Baixa Vacinação'
Municípios acima de 40 vacinações são 'Alta Vacinação'


In [6]:
df_export = (
    df_agregado

    .with_columns(
        pl.when(
            pl.col("total_vacinacoes") > q3
        )

        .then(
            pl.lit("Alta Vacinação")
        )

        .when(
            pl.col("total_vacinacoes") < q1
        )

        .then(
            pl.lit("Baixa Vacinação")
        )

        .otherwise(
            pl.lit("Média Vacinação")
        )

        .alias("Etiqueta_Classificacao")
    )
)

display(df_export.head(10))

municipio,uf,estado,regiao,total_vacinacoes,media_idade,Etiqueta_Classificacao
str,str,str,str,u32,f64,str
"""BELTERRA""","""PA""","""RJ""","""Sudeste""",6,46.67,"""Média Vacinação"""
"""GUAPO""","""GO""","""RJ""","""Sudeste""",5,35.2,"""Média Vacinação"""
"""GUARAI""","""TO""","""RJ""","""Sudeste""",14,29.36,"""Média Vacinação"""
"""PIRATININGA""","""SP""","""RJ""","""Sudeste""",10,45.9,"""Média Vacinação"""
"""GENTIO DO OURO""","""BA""","""RJ""","""Sudeste""",7,27.71,"""Média Vacinação"""
"""NOVO ARIPUANA""","""AM""","""RJ""","""Sudeste""",6,38.33,"""Média Vacinação"""
"""PIEDADE DO RIO GRANDE""","""MG""","""RJ""","""Sudeste""",27,36.56,"""Média Vacinação"""
"""BARIRI""","""SP""","""RJ""","""Sudeste""",13,29.54,"""Média Vacinação"""
"""FRECHEIRINHA""","""CE""","""RJ""","""Sudeste""",83,32.53,"""Alta Vacinação"""


In [7]:
df_export.write_csv(
    r"C:\Users\Usuario\Downloads\vacinas_rj_powerbi.csv"
)

print("Arquivo 'vacinas_rj_powerbi.csv' gerado com sucesso!")

Arquivo 'vacinas_rj_powerbi.csv' gerado com sucesso!
